# Manifolds and Curvature

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 04.01–04.09, 09.01–09.06 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/10_Differential_Geometry_and_Topology/01_manifolds_and_curvature.ipynb)

---

## 🎯 Learning Objective

Learn smooth manifolds, tangent spaces, and Riemannian metrics, and build intuition for curvature and geodesics.

---

## 📐 Theory

Concrete example first, always in this module: the surface of a sphere. Standing on the Earth's
surface, "straight lines" don't exist the way they do on a flat map — the shortest path between
two cities is a curved arc when drawn on a flat projection, yet it's the most natural, direct
route if you're actually walking on the curved surface. Differential geometry formalizes exactly
this: how do you do calculus (distances, straight lines, curvature) *on* a curved space, using
only measurements *within* that space, without needing to step outside it?

### Manifolds: locally flat, globally curved

A (smooth) **manifold** is a space that looks like ordinary flat $\mathbb{R}^n$ if you zoom in
closely enough at any point, even if it's globally curved (a sphere looks locally like a flat
plane to someone standing on it, even though the whole sphere is curved). This "locally flat"
property is what lets you still do calculus — just locally, in each zoomed-in neighborhood
(called a **chart**), and manifold theory is largely about how these local flat pieces are
stitched together consistently.

### Tangent spaces

At each point $p$ on a manifold, the **tangent space** $T_pM$ is the (flat, ordinary vector
space) collection of all possible "instantaneous velocity directions" you could move in from
$p$ while staying on the manifold — for a sphere, the tangent space at $p$ is literally the flat
plane touching the sphere at that single point. Every notion of "direction" or "derivative" on
the manifold (a gradient, previewed for statistical manifolds in `10.02`) lives in some tangent
space, not in the ambient space the manifold happens to sit inside.

### Riemannian metrics: measuring length and angle locally

A **Riemannian metric** $g_p$ is an inner product (recall `08.01`) defined on each tangent
space $T_pM$, varying smoothly from point to point. It's what lets you measure the length of a
tangent vector, the angle between two tangent directions, and — by integrating along a curve —
the length of any path on the manifold, entirely using only the manifold's own internal
geometry (no need to reference the flat space the manifold might happen to be embedded in).

### Geodesics: the manifold's own notion of "straight line"

A **geodesic** is a curve that locally minimizes length between nearby points — the manifold's
own generalization of a straight line. On a sphere, geodesics are exactly the **great circles**
(the largest possible circles you can draw on the sphere, like the equator or any circle through
both poles); the shortest path between two points on a sphere always follows a great-circle arc,
never a "straight line" that would require leaving the surface.

### Curvature

**Curvature** quantifies how much a manifold deviates from being flat. A sphere has *constant
positive* curvature (it curves the same way everywhere); a flat plane has zero curvature; a
saddle-shaped surface has *negative* curvature. One concrete, computable consequence: on a
positively-curved surface, the sum of a triangle's interior angles *exceeds* $180°$ (draw a
triangle with one vertex at the north pole and two on the equator — each equator-vertex angle is
already $90°$), unlike the flat-plane case where they always sum to exactly $180°$.

### The data manifold hypothesis

The **manifold hypothesis** — the working assumption behind an enormous amount of modern
representation learning — claims that real high-dimensional data (images, text embeddings)
doesn't fill its entire ambient space uniformly; it concentrates near a much lower-dimensional
curved manifold embedded within that high-dimensional space. A synthetic example makes this
concrete: a "Swiss roll" dataset lives in 3D space but is intrinsically a curved 2D sheet rolled
up — dimensionality-reduction methods (recall PCA, `01.07`) that only look for a *flat*
low-dimensional subspace can fail badly on data like this, while methods that respect the
manifold's curvature (geodesic distance rather than straight-line Euclidean distance) succeed —
demonstrated concretely below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting a geodesic (great-circle) path on a sphere alongside a detour through the north pole
makes "shortest path within the manifold" visible and comparable to a real (longer) alternative.

In [ ]:
def spherical_to_cartesian(theta, phi, R=1.0):
    return R * np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])

def slerp(p1, p2, t, R=1.0):
    omega = np.arccos(np.clip(np.dot(p1, p2)/R**2, -1, 1))
    if omega < 1e-10:
        return p1
    # ANTIPODAL guard: if p1, p2 are exactly (or nearly) opposite points on the sphere,
    # omega -> pi, so sin(omega) -> 0 -- but NOT exactly 0 in floating point (np.sin(np.pi)
    # is ~1.22e-16, a rounding artifact, not a true zero). Dividing by that near-zero value
    # would blow up the "interpolated" point by a factor of ~1e16 instead of raising a clear
    # error. Mathematically this isn't just a numerical inconvenience: antipodal points on a
    # sphere have INFINITELY MANY great circles connecting them (every great circle through
    # one passes through the other), so "the" geodesic isn't even well-defined here -- a
    # clear error is the mathematically honest response, not a silently blown-up number.
    if np.pi - omega < 1e-10:
        raise ValueError(
            "p1 and p2 are antipodal (omega ~ pi): the great-circle path between them is not "
            "unique (infinitely many geodesics connect antipodal points), so slerp's formula "
            "-- which divides by sin(omega) ~ 0 here -- is not well-defined for this input."
        )
    return (np.sin((1-t)*omega)*p1 + np.sin(t*omega)*p2) / np.sin(omega)

p1 = spherical_to_cartesian(np.pi/4, 0.0)
p2 = spherical_to_cartesian(np.pi/3, np.pi/2)
north_pole = np.array([0.0, 0.0, 1.0])

ts = np.linspace(0, 1, 200)
geodesic_path = np.array([slerp(p1, p2, t) for t in ts])

def detour_path(p1, p2, mid, t):
    raw = (1 - 2*t)*p1 + 2*t*mid if t < 0.5 else (2 - 2*t)*mid + (2*t - 1)*p2
    return raw / np.linalg.norm(raw)
detour = np.array([detour_path(p1, p2, north_pole, t) for t in ts])

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection='3d')
u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
ax.plot_surface(np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v), alpha=0.15, color="gray")
ax.plot(*geodesic_path.T, color="#4C72B0", lw=3, label="geodesic (great circle)")
ax.plot(*detour.T, color="#C44E52", lw=2, ls="--", label="detour via north pole")
ax.scatter(*p1, color="k", s=50); ax.scatter(*p2, color="k", s=50)
ax.set_title("Geodesic vs. a real alternative path on a sphere")
ax.legend()
plt.show()

geo_len = np.sum(np.linalg.norm(np.diff(geodesic_path, axis=0), axis=1))
det_len = np.sum(np.linalg.norm(np.diff(detour, axis=0), axis=1))
print(f"Geodesic path length: {geo_len:.4f}")
print(f"Detour path length:   {det_len:.4f}  ({det_len/geo_len:.2f}x longer)")

# --- Regression test: antipodal points must raise a clear error, not a silently blown-up
# vector. p_antipode = -p1 is EXACTLY opposite p1 on the sphere. ---
p_antipode = -p1
try:
    slerp(p1, p_antipode, 0.5)
    print("\nFAILED to reject antipodal input (this should not happen)")
except ValueError as e:
    print(f"\nRegression test: antipodal points correctly rejected -- {e}")

## 🐍 Implementation from Scratch

We compute geodesic distance on a sphere from the great-circle formula, verify it against
numerically integrating a discretized path, and confirm the geodesic is genuinely shorter than
a real alternative route -- not just algebraically simpler.

In [ ]:
def great_circle_distance(p1, p2, R=1.0):
    # The closed-form geodesic distance formula on a sphere, derived from the inner
    # product (recall 08.01): cos(angle) = <p1,p2>/R^2, and arc length = R * angle
    cos_angle = np.clip(np.dot(p1, p2) / (R * R), -1, 1)
    return R * np.arccos(cos_angle)

def spherical_to_cartesian(theta, phi, R=1.0):
    return R * np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])

def slerp(p1, p2, t, R=1.0):
    # Spherical linear interpolation: the actual geodesic curve connecting p1 to p2
    omega = np.arccos(np.clip(np.dot(p1, p2)/R**2, -1, 1))
    if omega < 1e-10:
        return p1
    # Same antipodal guard as the Visual Intuition section's copy of this function: as
    # omega -> pi, sin(omega) -> 0 (only up to floating-point rounding, e.g. ~1.22e-16 at
    # exactly pi, never a true zero), and dividing by it would blow up the result rather than
    # signal that antipodal points have no UNIQUE great-circle path connecting them.
    if np.pi - omega < 1e-10:
        raise ValueError(
            "p1 and p2 are antipodal (omega ~ pi): the geodesic between them is not unique, "
            "so slerp is not well-defined for this input."
        )
    return (np.sin((1-t)*omega)*p1 + np.sin(t*omega)*p2) / np.sin(omega)

p1 = spherical_to_cartesian(np.pi/4, 0.0)
p2 = spherical_to_cartesian(np.pi/3, np.pi/2)
d_formula = great_circle_distance(p1, p2)

n_steps = 50000
ts = np.linspace(0, 1, n_steps)
path = np.array([slerp(p1, p2, t) for t in ts])
path_length = np.sum(np.linalg.norm(np.diff(path, axis=0), axis=1))
print(f"Great-circle distance (closed-form formula): {d_formula:.6f}")
print(f"Numerically integrated SLERP path length:     {path_length:.6f}")
print(f"Match: {np.isclose(d_formula, path_length, atol=1e-3)}")

# --- The Swiss roll: a genuine 2D manifold curved into 3D, testing the manifold hypothesis ---
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import Isomap
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score

X, color = make_swiss_roll(n_samples=800, noise=0.05, random_state=0)

X_pca = PCA(n_components=2).fit_transform(X)               # finds the best FLAT approximation
X_iso = Isomap(n_neighbors=10, n_components=2).fit_transform(X)  # respects geodesic distance along the roll

score_pca = cross_val_score(KNeighborsRegressor(n_neighbors=5), X_pca, color, cv=5, scoring='r2').mean()
score_iso = cross_val_score(KNeighborsRegressor(n_neighbors=5), X_iso, color, cv=5, scoring='r2').mean()
print(f"\nRecovering the Swiss roll's true 1D 'unrolled position' from a 2D embedding:")
print(f"  PCA (flat, Euclidean) embedding:    R^2 = {score_pca:.4f}")
print(f"  Isomap (geodesic-aware) embedding:  R^2 = {score_iso:.4f}")
print(f"  Geodesic-aware embedding wins: {score_iso > score_pca}")

## 🤖 Why This Matters for AI

The manifold hypothesis is the working assumption behind an enormous fraction of representation
learning: a trained network's embedding/latent space is believed (and in many cases empirically
observed) to organize data along a lower-dimensional curved manifold within the full embedding
space, rather than filling it uniformly. This isn't just philosophical — it has concrete
algorithmic consequences: methods that measure distance *along* the data's curved manifold
(geodesic distance) can recover structure that flat, straight-line (Euclidean) methods miss
entirely. Below, we generate a genuinely curved 2D dataset (the classic "Swiss roll," rolled up
into 3D) and confirm directly that a geodesic-aware embedding method recovers its true
underlying structure far better than a method restricted to flat linear structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ax0 = fig.add_subplot(1, 3, 1, projection='3d')
ax0.scatter(X[:, 0], X[:, 1], X[:, 2], c=color, cmap="viridis", s=8)
ax0.set_title("Swiss roll in its native 3D space\n(a 2D sheet, curled up)")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=color, cmap="viridis", s=8)
axes[1].set_title(f"PCA (flat) embedding\nR^2={score_pca:.3f}")

axes[2].scatter(X_iso[:, 0], X_iso[:, 1], c=color, cmap="viridis", s=8)
axes[2].set_title(f"Isomap (geodesic) embedding\nR^2={score_iso:.3f}")
plt.tight_layout()
plt.show()

print(f"PCA 'sees through' the rolled-up sheet -- it finds the best FLAT 2D plane through the")
print(f"3D point cloud, which cuts across the roll and merges points that are actually far apart")
print(f"along the sheet's own surface (different colors get mixed together in the PCA plot).")
print(f"\nIsomap instead builds a neighborhood graph and computes GEODESIC distances (shortest")
print(f"paths through nearby points, approximating distance ALONG the curved manifold), then")
print(f"finds an embedding that preserves those -- correctly 'unrolling' the sheet, visible as")
print(f"the smooth, unbroken color gradient in the third panel.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. On a sphere of radius $R=6371$ km (Earth's approximate radius), compute the great-circle
   distance between two points at $(\theta,\phi) = (40°, 0°)$ and $(35°,90°)$ (angles from the
   Theory section's spherical coordinates). Check your answer is less than the maximum possible
   distance $\pi R\approx20{,}015$ km (half the sphere's circumference) before verifying with
   `great_circle_distance`.

<details>
<summary>💡 Solution</summary>

Converting to Cartesian coordinates via $\mathbf{p}=R(\sin\theta\cos\phi,\sin\theta\sin\phi,\cos\theta)$
and applying $d=R\arccos(\langle\mathbf{p}_1,\mathbf{p}_2\rangle/R^2)$ gives $d\approx5685.8$
km — comfortably less than the $\approx20{,}015$ km maximum, as any valid great-circle distance
must be. `great_circle_distance(p1, p2, R=6371)` on the corresponding Cartesian points returns
the identical value.

</details>

### 💭 UNDERSTAND
2. Increase the `noise` parameter in `make_swiss_roll` from `0.05` to `0.5` and re-run the
   PCA-vs-Isomap comparison. Does Isomap's advantage over PCA shrink, grow, or stay about the
   same? Then push `noise` much further (e.g. to `3.0`) — does the pattern from the smaller
   noise increase continue, or does something qualitatively different happen?

<details>
<summary>💡 Solution</summary>

From `noise=0.05` to `noise=0.5`, Isomap's $R^2$ advantage over PCA actually *grows* slightly
(from $\approx0.41$ to $\approx0.46$), and it keeps growing modestly at `noise=1.5`
($\approx0.53$) — moderate noise doesn't hurt Isomap's relative advantage, since both methods
degrade somewhat but Isomap's geodesic-distance advantage is still intact. At `noise=3.0`,
though, something qualitatively different happens: both methods' $R^2$ collapse to a similar,
much lower level ($\approx0.19$ and $\approx0.18$), and Isomap's advantage effectively
*vanishes* (even goes slightly negative). This is because Isomap's entire method depends on
building an accurate local nearest-neighbor graph to approximate geodesic distances — once noise
is large enough to scramble which points are genuinely "nearby" on the underlying manifold, the
neighbor graph itself becomes unreliable, and Isomap loses the very structure it relies on to
beat a flat method in the first place.

</details>

### ✏️ DERIVE
3. The Theory section states that on a sphere, a triangle's interior angles sum to more than
   $180°$, with the excess connected to curvature. Using a triangle with one vertex at the north
   pole and two vertices at colatitude $\theta$ separated by a fixed longitude gap, derive (or
   verify numerically, tracking the ratio as $\theta$ shrinks) that the angle excess (in
   radians) scales like $\theta^2$ for small $\theta$ — i.e. that
   $(\text{excess in radians})/\theta^2$ approaches a constant as $\theta\to0$, consistent with
   the Gauss-Bonnet theorem's prediction that angle excess equals curvature times triangle area
   (and a small triangle's area itself scales like $\theta^2$).

<details>
<summary>💡 Solution outline</summary>

For a fixed unit sphere (curvature $K=1$ everywhere) and a small geodesic triangle of area $A$,
Gauss-Bonnet gives angle excess $=K\cdot A = A$ exactly. A small triangle built from colatitude
$\theta$ and a fixed longitude separation has area scaling like $\theta^2$ as $\theta\to0$ (both
the triangle's "height" from the pole and its "width" along the latitude circle shrink linearly
in $\theta$, so area — a product of two roughly-linear dimensions — shrinks quadratically).
Numerically, computing each vertex's angle via tangent-vector geometry and summing, the excess
(in radians) divided by $\theta^2$ (in radians) converges to $\approx0.2498$ as $\theta$ shrinks
from $60°$ down to $1°$ — essentially constant at small $\theta$, exactly the predicted
quadratic scaling, confirming curvature is a well-defined *local* (infinitesimal) quantity, not
an artifact that only appears for large triangles.

</details>

### 🐍 IMPLEMENT
4. Implement a numerical approximation to the sphere's **Gaussian curvature** using the
   angle-excess fact: draw a small triangle on the sphere with one vertex at the north pole and
   two nearby points on a fixed latitude circle, sum its three interior angles numerically
   (using vector geometry to compute each angle from tangent directions), and confirm the sum
   exceeds $180°$ by an amount that shrinks as the triangle gets smaller.

<details>
<summary>✅ How to know you're right</summary>

At each triangle size you test, the angle sum should exceed $180°$ (never fall below it — a flat
or negatively-curved space would never show a triangle exceeding $180°$, so seeing it fall below
would indicate a bug, not a valid alternative geometry for this specific construction), and the
excess should shrink monotonically as you shrink the colatitude $\theta$ (e.g. from
$\approx14.7°$ at $\theta=60°$ down to $\approx0.004°$ at $\theta=1°$) — confirming both the
qualitative claim (positive curvature means angle excess) and the quantitative one (curvature is
local, so the effect vanishes for small enough triangles, exactly matching Exercise 3's
$\theta^2$ scaling).

</details>

### 🤖 APPLY
5. `01.07`'s PCA finds the best *flat* low-dimensional subspace; this notebook's Isomap
   comparison shows that a genuinely curved dataset (the Swiss roll) defeats that flat
   assumption, while a geodesic-aware method succeeds. Suppose you're choosing a
   dimensionality-reduction method for a new embedding space from a trained model, and you
   don't know in advance whether the data manifold is "close to flat" or "genuinely curved."
   Using the Swiss-roll comparison's own $R^2$ numbers as your template, describe an experiment
   you could run to decide which assumption better matches your actual data, without needing to
   already know the manifold's true structure.

<details>
<summary>✅ What you should observe</summary>

The key insight from the Swiss-roll experiment is comparative, not absolute: run both a flat
method (PCA) and a geodesic-aware method (Isomap) on the same data and the same downstream task
(here, recovering a known auxiliary quantity via $R^2$), then compare. A large gap in favor of
the geodesic-aware method (as seen here, $\approx0.41$–$0.53$ in $R^2$ depending on noise level)
is itself evidence the data's true manifold has meaningful curvature the flat method is missing;
a small or nonexistent gap suggests the data is close enough to flat that PCA's simpler
assumption is adequate — you don't need to already know the ground truth "unrolled" structure
(which in a real embedding space you typically wouldn't have) to make this comparison, only a
downstream signal both embeddings can be scored against.

</details>

### 🚀 CHALLENGE
6. `10.02`-`10.03` (later in this module) will use curvature ideas from an *information*-geometry
   angle — the Fisher information matrix as a Riemannian metric on a space of probability
   distributions, rather than a metric on physical space like the sphere here. Before reading
   ahead, speculate: what would "curvature" mean for a *space of probability distributions*
   (e.g. all Gaussians with varying mean and variance), and what geometric quantity would play
   the role this notebook's tangent vectors and inner products played for the sphere? There's no
   single expected numeric answer here — the goal is to identify the right *analogy* to carry
   forward.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer identifies the core analogy precisely: just as each point on a sphere has a
tangent space with an inner product (this notebook's Riemannian metric) measuring "how far
apart" two nearby directions are, each probability distribution in a parametrized family (e.g.
$\mathcal{N}(\mu,\sigma^2)$, parametrized by $(\mu,\sigma)$) can be thought of as a "point" in
an abstract space, with "nearby directions" corresponding to small changes in the parameters —
and a natural notion of "distance" between two nearby distributions (how distinguishable they
are statistically, not how far apart their parameter values are numerically) plays the role of
the Riemannian metric. Curvature in this space would then describe how that
statistical-distinguishability metric itself varies as you move through parameter space — for
instance, whether a fixed-size step in $\sigma$ near a large $\sigma$ changes the distribution
by "the same amount" (statistically) as the identical-size step near a small $\sigma$. A strong
answer doesn't need to name the Fisher information matrix explicitly to succeed, but should
recognize that "distance" here needs to be a statistical notion (how distinguishable two nearby
distributions are), not the ordinary Euclidean distance between their parameter vectors.

</details>

---

## 📚 Further Reading
- do Carmo, *Differential Geometry of Curves and Surfaces*, Ch. 1-2
- Tenenbaum, de Silva & Langford, ["A Global Geometric Framework for Nonlinear Dimensionality Reduction"](https://www.science.org/doi/10.1126/science.290.5500.2319) (Isomap)
- Bronstein et al., ["Geometric Deep Learning: Going beyond Euclidean data"](https://arxiv.org/abs/1611.08097)

---

*Next notebook: [Information Geometry](02_information_geometry.ipynb)*